## Shannon invariants: A scalable approach to information decomposition
Aaron J. Gutknecht, Fernando E. Rosas, David A. Ehrlich, Abdullah Makkeh, Pedro A. M. Mediano, Michael Wibral
Supplementary Code -  Script 1/2
### Training of the 8-level-quantized, one-hot-output MNIST network and computation of Shannon invariant summary structures (Figure 2)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
cm = 1/2.54  # centimeters to inches

import nninfo
from nninfo.plot import format_figure_broken_axis, plot_mean_and_interval

### Train network

In [ ]:
# Set experiment id
experiment_id = "mnist_8levels_onehot"

#### Set up parameters and train first network

In [ ]:
# Note that we do not set initial seeds manually here, but save all seeds to the
# checkpoints files during training for later reproducibility. Rerunning this script
# will produce slightly different figures due to the randomness of network
# initialization etc.

layer_infos = [
    nninfo.LayerInfo(connection_layer='input', activation_function='input'),
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': 784, 'out_features': 50}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': 50, 'out_features': 10}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': 10, 'out_features': 5}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': 5, 'out_features': 5}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': 5, 'out_features': 5}, activation_function='tanh'),
    nninfo.LayerInfo(connection_layer='linear', connection_layer_kwargs={'in_features': 5, 'out_features': 10}, activation_function='softmax_output')
]

# Set weight initialization
initializer_name = 'xavier'

# Create network instance
network = nninfo.NeuralNetwork(layer_infos=layer_infos, init_str=initializer_name)

# Set task instance
task = nninfo.TaskManager('mnist_1d_dat')
# Split dataset into MNIST training set, and MNIST+QMNIST test set
task['full_set'].train_test_val_sequential_split(60_000, 60_000, 0)

# Create quantizer list with stochastic quantization. The input layer is not quantized.
quantizer = [None] + 5 * [{'levels': 8, 'rounding_point': 'stochastic'}] + [None]

#Set up trainer
trainer = nninfo.Trainer(dataset_name='full_set/train',
                        optim_str='SGD',
                        loss_str='CELoss',
                        lr=0.01,
                        shuffle=True,
                        batch_size=64,
                        quantizer=quantizer)

# Set up tester
tester = nninfo.Tester(dataset_name='full_set/test')

# Set up schedule
schedule = nninfo.Schedule()
# Save training state for 50 logarithmically spaced checkpoints
schedule.create_log_spaced_chapters(100_000, 50)

# Combine components into experiment
experiment = nninfo.Experiment(experiment_id=experiment_id,
                        network=network,
                        task=task,
                        trainer=trainer,
                        tester=tester,
                        schedule=schedule)

# Run training for 10^5 epochs
experiment.run_following_schedule()

#### Rerun training with different random weight initializations

In [ ]:
# Set up experiment
experiment = nninfo.Experiment.load(experiment_id)

# Compute 19 more training runs with different random weight initializations.
experiment.rerun(19)

### Evaluate network performance
#### Compute loss and accuracy

In [ ]:
quantizer_params = [None] + 5 * [{'levels': 8, 'rounding_point': 'stochastic'}] + [None]

experiment = nninfo.Experiment.load(experiment_id)
performance_measurement = nninfo.analysis.PerformanceMeasurement(experiment, ['full_set/train', 'full_set/test'], quantizer_params=quantizer_params)

performance_measurement.perform_measurements(run_ids='all', chapter_ids='all', exists_ok=True)

#### Plot loss and accuracy

In [ ]:
# Load performance file
experiment = nninfo.Experiment.load(experiment_id)
performance_measurement = nninfo.analysis.PerformanceMeasurement.load(experiment, "performance")

fig, ax = plt.subplots(figsize=(4*cm, 4*cm), dpi=150)
ax.set_ylim(0, 1)
twinax = ax.twinx()

# Plot accuracy
nninfo.plot.plot_loss_accuracy(performance_measurement.results, ax, twinax)

ax.legend(ncol=1, bbox_to_anchor=(1.5, 0.5), loc='center left');

In [ ]:
# Save result
plt.savefig(f"experiments/exp_{experiment_id}/plots/performance.pdf", bbox_inches='tight')

### Compute Shannon invariant summary structures on $L_3$, $L_4$ and $L_5$

In [ ]:
# Load experiment
experiment = nninfo.Experiment.load(experiment_id)

for layer in [3, 4, 5]:

    target = [nninfo.NeuronID('Y', tuple())]
    source1 = [nninfo.NeuronID(f'L{layer}', (1,))]
    source2 = [nninfo.NeuronID(f'L{layer}', (2,))]
    source3 = [nninfo.NeuronID(f'L{layer}', (3,))]
    source4 = [nninfo.NeuronID(f'L{layer}', (4,))]
    source5 = [nninfo.NeuronID(f'L{layer}', (5,))]

    quantization_dict = [None] + 5 * [{'levels': 8, 'rounding_point': 'stochastic'}] + [None]

    measurement = nninfo.analysis.PIDMeasurement(experiment=experiment,
                                                 measurement_id=f'summary_measurement_L{layer}',
                                                 dataset_name='full_set/train',
                                                 quantizer_params=quantization_dict,
                                                 pid_definition='deg_red',
                                                 binning_kwargs={'binning_method' : 'none'},
                                                 target_id_list=target,
                                                 source_id_lists=[source1, source2, source3, source4, source5])

    itic = time.time_ns()
    measurement.perform_measurements(run_ids='all', chapter_ids='all', exists_ok=True)
    itoc = time.time_ns()

    print(f"Computing summary structures for L{layer} took: ", (itoc-itic)/10**9, "s")

#### Compute degree of redundancy and vulnerability from measurements and plot

In [ ]:
def compute_degred(df):
    """ Compute average degree reduction for each column containing the mutual information with an individual source.
    """

    df['avg_deg_red'] = df.filter(regex='^source_MI[0-9]*$', axis=1).sum(axis=1) / df['total_MI']

    return df

def compute_vuln(df):
    """ Compute vulnerability for each column containing the mutual information with all but one source.
    """

    num_sources = len(df.filter(regex='^allbut_source_MI[0-9]*$', axis=1).columns)
    df['vuln'] = (num_sources*df['total_MI'] - df.filter(regex='^allbut_source_MI[0-9]*$', axis=1).sum(axis=1)) / df['total_MI']

    return df

In [ ]:
# Plot degree of redundancy for each layer

fig, ax= plt.subplots(figsize=(5*cm, 4*cm), dpi=150)
format_figure_broken_axis(ax)

experiment = nninfo.experiment.Experiment.load(experiment_id)
colors = ['tab:blue', 'tab:orange', 'tab:green']
for layer, color in zip([3, 4, 5], colors):
    summary_measurement = nninfo.analysis.PIDMeasurement.load(experiment=experiment, measurement_id=f'summary_measurement_L{layer}')
    summary_df = compute_degred(summary_measurement.results)
    plot_mean_and_interval(summary_df, ax, x='epoch_id', y='avg_deg_red', label=f'Layer {layer}', color=color)

ax.set_xlabel('Epoch')
ax.set_ylabel('Deg. of Redundancy')

ax.legend(bbox_to_anchor=(1, .6), loc='upper left')

# Save results
plt.savefig(f"../experiments/exp_{experiment_id}/plots/degree_of_redundancy.pdf", bbox_inches='tight')

In [ ]:
# Plot degree of vulnerability for each layer

fig, ax= plt.subplots(figsize=(5*cm, 4*cm), dpi=150)
format_figure_broken_axis(ax)

experiment = nninfo.experiment.Experiment.load(experiment_id)
colors = ['tab:blue', 'tab:orange', 'tab:green']
for layer, color in zip([3, 4, 5], colors):
    summary_measurement = nninfo.analysis.PIDMeasurement.load(experiment=experiment, measurement_id=f'summary_measurement_L{layer}')
    summary_df = compute_vuln(summary_measurement.results)
    plot_mean_and_interval(summary_df, ax, x='epoch_id', y='vuln', label=f'Layer {layer}', color=color)

ax.set_xlabel('Epoch')
ax.set_ylabel('Deg. of Vulnerability')

ax.legend(bbox_to_anchor=(1, .6), loc='upper left')

# Save results
plt.savefig(f"../experiments/exp_{experiment_id}/plots/degree_of_vulnerability.pdf", bbox_inches='tight')